# ByteCore-DualMemory — 4-Hour Pre-Training (T4)

**Phase 24 pipeline, end to end on a free Colab T4.**

- **Model:** 8-layer, 1024-dim byte Transformer (274,025,753 params) with a
  **Gated Skill Adapter** (1024 ↔ 8192 GLU bridge) in every layer, a
  confidence-headed adaptive refinement loop (≤4 passes, early break at 0.9)
  over a **Latent Thought Matrix** — an isolated (B, 4, 1024) scratchpad with
  bidirectional cross-attention that never emits bytes (Phase 24.4) —
  and frozen multimodal encoders projecting into the 8192-dim skill bank.
- **Data mix (streaming, PT-BR first):** **50% Portuguese (Brazil)**
  (`TucanoBR/GigaVerbo` → `fineweb-2` por_Latn) · 25% code (`the-stack`
  Python/JS) · 15% multimodal (COYO images + LibriSpeech audio, bidirectional
  byte prediction) · 10% English (`fineweb`). The quota is enforced per
  100-draw window, not on average, and cell 6 audits it before the run.
  **100% streaming (Phase 24.2):** every source is a verified
  `IterableDataset` with a RAM-only shuffle buffer — cell 6b proves
  **zero disk growth** before any training starts.
- **VRAM:** bf16 on Ampere+; **fp16 + GradScaler on T4** (sm_75 cannot bf16).
  Gradient checkpointing, `batch_size=4`, `grad_accum=8`, measured VRAM probe
  with automatic batch shrink.
- **Checkpoints:** atomic, every **30 minutes**, pushed to
  `gabrielbosse9/airesearch` on the Hugging Face Hub. Re-running this notebook
  resumes from the latest checkpoint automatically.

> Runtime: **Runtime → Change runtime type → T4 GPU** before starting.
> Add your Hugging Face token as the Colab secret `HF_TOKEN` (write access to
> `gabrielbosse9/airesearch`), or enter it interactively in the auth cell.
> Total wall clock ≈ 4 h: ~10 min setup + 220 min training + eval/upload.

In [ ]:
# 1) GPU check — fail fast if the runtime is not a GPU.
import torch

assert torch.cuda.is_available(), "Enable GPU: Runtime > Change runtime type > T4 GPU"
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name} (sm_{major}{minor}), {total_gb:.1f} GB")
print("bf16 supported:", torch.cuda.is_bf16_supported())
if not torch.cuda.is_bf16_supported():
    print("T4 detected -> trainer resolves to float16 + GradScaler (required fallback)")

In [ ]:
# 2) Dependencies. torch/torchvision/torchaudio ship with Colab; add the rest.
import subprocess
import sys

packages = ["datasets>=3,<6", "huggingface_hub>=0.25", "pyyaml", "ddgs",
            "nbformat>=5", "tqdm", "pillow"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("dependencies ready")

In [ ]:
# 3) Repository. Uses the merged Phase 24.1 main branch (PT-BR first).
import os
import subprocess

REPO_URL = "https://github.com/gabrielbosse1/airesearch.git"
REPO_DIR = "/content/airesearch"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
print("working directory:", os.getcwd())

In [ ]:
# 4) Hugging Face auth (checkpoints every 30 min go to gabrielbosse9/airesearch).
token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass

from huggingface_hub import login
if token:
    login(token=token, add_to_git_credential=False)
else:
    login()  # interactive widget fallback
print("Hugging Face auth ready")

In [ ]:
# 5) Configuration + exact parameter count (meta device: zero memory cost).
import torch
import yaml

from models.foundation import CausalByteTransformer, FoundationConfig

with open("configs/t4_scaled.yaml", "r", encoding="utf8") as handle:
    config = yaml.safe_load(handle)
cfg = FoundationConfig.from_mapping(config["model"])
print("layers:", cfg.n_layers, "| d_model:", cfg.d_model, "| context:", cfg.max_seq_len)
print("memory_dim:", cfg.memory_dim, "| refinement:", cfg.refinement_steps,
      "| confidence threshold:", cfg.confidence_threshold)
with torch.device("meta"):
    probe = CausalByteTransformer(cfg)
print(f"parameters: {probe.parameters_count:,}")
print("data weights:", config["data"]["weights"])

## Pre-flight: prove the PT-BR-first data mix

Portuguese (Brazil) is the primary language of the model, so the stream is
**50% PT-BR documents / 25% code / 15% multimodal / 10% English**.

`data/sources.MixtureSampler` allocates each **window of 100 draws** with the
largest-remainder method and shuffles it with the seeded RNG: 100 draws are
*exactly* 50/25/15/10, not merely on average. The cell below pulls **100 real
batches** through the streaming mixture and counts where every document came
from, so a broken source chain (e.g. COYO dead links) is visible **before** the
4-hour GPU budget is spent.

Every stream is shuffled with the iterable-safe
`.shuffle(seed=..., buffer_size=10000)` — a **RAM-only reservoir** (media
fallbacks capped at 1,000 decoded rows), never the map-style
`.shuffle(seed=...)` that fails on an `IterableDataset`. The
zero-disk cell after the audit then proves nothing is cached to disk.

Two numbers are printed, and they are different quantities: the **document
share** (the contract, which must match the weights) and the **byte share**
(informational — a Python file is far longer than an image caption, so bytes
will never match the document quota).

In [ ]:
# 6) PT-BR-first mix audit -- 100 real batches, before any training starts.
from data.sources import audit_mixture, build_mixture, format_mixture_report

train_cfg, data_cfg = config["training"], config["data"]
mix = build_mixture(data_cfg, seed=int(train_cfg["seed"]))
report = audit_mixture(
    mix,
    batches=int(train_cfg.get("mix_audit_batches", 100)),
    batch_size=int(train_cfg["micro_batch_size"]),
    tolerance=float(train_cfg.get("mix_audit_tolerance", 0.02)),
    max_seconds=180,                      # never hang the notebook on a slow chain
)
print(format_mixture_report(report))
print("effective weights:", mix.weights)
if report["failures"]:
    print("dropped source chains:", report["failures"])
assert report["quotas_ok"], "mix deviates from 50/25/15/10 -- fix the source chains above"
assert report["sample_share"]["ptbr"] >= 0.48, "PT-BR must dominate the stream"
print(f"OK: {report['samples']} documents, PT-BR share "
      f"{report['sample_share']['ptbr'] * 100:.1f}% (target 50.0%)")

## Zero-Disk verification (Phase 24.2)

`the-stack`, GigaVerbo and fineweb are **massive** — downloading them would
fill the Colab disk and crash the instance. The pipeline is therefore
**100% streaming**: `data/sources.open_stream` forces `streaming=True`,
verifies every source is an `IterableDataset` (a disk-backed `Dataset`
raises `TypeError` immediately), and shuffles with a RAM-only
`buffer_size=10000` reservoir (embedded-image/audio fallbacks are capped at
1,000 rows — decoded media rows are 100–1000× larger than text). The training loop is strictly **step-based**
(`while step < max_steps`) — `len(dataset)` and epochs never appear.

The cell below proves it on the live streams **before any training**:
it prints each group's dataset type, then measures disk usage before and
after the first **10 streamed batches** and asserts the growth is negligible
(≤ 32 MB — streaming metadata is KB-scale; a wholesale download is GB-scale).

In [ ]:
# 6b) ZERO-DISK verification (Phase 24.2): prove 100% streaming before training.
#     The Colab disk must NEVER fill with dataset files: every source must be
#     an IterableDataset, and 10 streamed batches must not grow disk usage.
import shutil

from datasets import IterableDataset

from data.sources import (DEFAULT_SOURCES, SHUFFLE_BUFFER_SIZE, MixtureSampler,
                          effective_shuffle_buffer, open_stream)

seed = int(train_cfg["seed"])
buffer_size = int(data_cfg.get("shuffle_buffer_size", SHUFFLE_BUFFER_SIZE))

# 1) Open each group's chain and print the stream TYPE (must be IterableDataset).
for group, specs in DEFAULT_SOURCES.items():
    if group not in mix.weights:
        print(f"{group:>10}: dropped by the audit above -- skipped")
        continue
    for spec in specs:
        try:
            stream = open_stream(spec, seed=seed, shuffle_buffer=buffer_size)
        except Exception as exc:
            print(f"{group:>10}: {spec.id} unavailable ({type(exc).__name__}) -- fallback")
            continue
        effective = effective_shuffle_buffer(spec, buffer_size)
        print(f"{group:>10}: {spec.id} -> {type(stream).__name__} "
              f"(shuffle buffer_size={effective:,})")
        assert isinstance(stream, IterableDataset), (
            f"{spec.id} returned {type(stream).__name__}: streaming=True violated")
        assert not hasattr(stream, "__len__"), "streaming datasets must not be sized"
        break
    else:
        raise AssertionError(f"group {group!r}: no streaming source could be opened")

assert isinstance(mix, MixtureSampler) and not hasattr(mix, "__len__")

# 2) Disk usage before/after the FIRST 10 streamed batches (no training yet).
def disk_used_mb() -> float:
    usage = shutil.disk_usage("/")   # Colab: /content and the HF cache live here
    return (usage.total - usage.free) / 1e6

before_mb = disk_used_mb()
docs = 0
for _ in range(10):
    docs += len(mix.next_batch(int(train_cfg["micro_batch_size"])))
after_mb = disk_used_mb()
growth_mb = after_mb - before_mb
print(f"disk used: {before_mb:,.1f} MB -> {after_mb:,.1f} MB "
      f"(growth {growth_mb:+.2f} MB over 10 batches / {docs} documents)")

# 3) Streaming metadata/lock files are KB-scale; a wholesale download is GB-scale.
MAX_GROWTH_MB = 32.0
assert growth_mb <= MAX_GROWTH_MB, (
    f"disk grew {growth_mb:.1f} MB while streaming 10 batches -- data is being "
    "cached to disk; STOP and fix the pipeline before the 4-hour run")
print("OK: zero-disk streaming verified (IterableDataset types + negligible growth)")

## Training

One unified script: `python -m training.train`. It streams the PT-BR-first 50/25/15/10 mix
(quota-exact per 100 draws; the cell above already proved it),
builds **bidirectional multimodal batches** (caption bytes ⇄ media codec bytes),
and optimises causal-LM cross-entropy + the auxiliary confidence BCE.

- Checkpoints: `runs/checkpoints/checkpoint_latest.pt`, every 30 min + every 500
  steps, uploaded to `gabrielbosse9/airesearch/checkpoints/checkpoint_latest.pt`.
- The VRAM probe measures a real forward+backward before step 0 and shrinks the
  micro-batch 4 → 2 → 1 if the peak exceeds 12 GB.
- If Colab disconnects, just re-run the notebook: training resumes from the
  latest local or Hub checkpoint.

In [ ]:
# 7) The 4-hour run (220-minute training budget inside the ~4 h session).
import subprocess
import sys
import time

HF_REPO = "gabrielbosse9/airesearch"
cmd = [sys.executable, "-u", "-m", "training.train",
       "--config", "configs/t4_scaled.yaml",
       "--hf-repo", HF_REPO,
       "--max-minutes", "220",
       "--checkpoint-minutes", "30"]
print("launching:", " ".join(cmd), flush=True)
start = time.time()
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
    exit_code = proc.wait()
print(f"training exit code {exit_code} after {(time.time() - start) / 60:.1f} min")
assert exit_code == 0, "training failed -- inspect the log above"


In [ ]:
# 8) Post-training sanity: generation, adaptive loop stats, true tooling demo.
import torch

from models.foundation import (CausalByteTransformer, FoundationConfig,
                               decode_ids, encode_bytes)
from training.train import make_search_fn

ckpt = torch.load("runs/checkpoints/checkpoint_latest.pt", map_location="cuda",
                  weights_only=False)
model = CausalByteTransformer(FoundationConfig(**ckpt["config"])).cuda()
model.load_state_dict(ckpt["model"])
model.eval()
print("resumed checkpoint at step", ckpt["step"])

prompt = encode_bytes(b"The capital of Brazil is").cuda()
sample = model.generate(prompt, max_new_bytes=64)
print("sample:", decode_ids(sample))

_, info = model(prompt.unsqueeze(0), return_info=True)
print("adaptive loop:", {"steps_run": info["steps_run"], "confidence": info["confidence"]})
# Phase 24.4: the latent scratchpad is purely internal -- report its usage.
thoughts = model.refinement.thoughts if model.refinement is not None else None
if thoughts is not None:
    print("latent thought buffer:", {
        "shape": list(thoughts.last_buffer.shape),
        "thought_steps": info["thought_steps"],
        "activation_norm": round(info["thought_norm"], 4),
        "buffer_gate": round(float(thoughts.buffer_gate.detach()), 4),
        "context_gate": round(float(thoughts.context_gate.detach()), 4)})

# [TOOL_SEARCH] token ids trigger a real sandboxed search (mock backend here;
# pass make_search_fn("ddgs") for live DuckDuckGo on this networked runtime).
ids, stats = model.generate_with_tools(encode_bytes(b"search: ").cuda(),
                                       make_search_fn("mock"), max_new_bytes=96)
print("tool demo:", stats)
print("transcript:", decode_ids(ids, keep_controls=True)[:300])

In [ ]:
# 9) Final artifacts to the Hub (the trainer already pushed the latest ckpt).
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(HF_REPO, repo_type="model", exist_ok=True)
artifacts = [
    ("runs/checkpoints/checkpoint_latest.pt", "checkpoints/checkpoint_final.pt"),
    ("configs/t4_scaled.yaml", "configs/t4_scaled.yaml"),
    ("runs/metrics.jsonl", "logs/metrics.jsonl"),
]
for local_path, repo_path in artifacts:
    try:
        api.upload_file(path_or_fileobj=local_path, path_in_repo=repo_path,
                        repo_id=HF_REPO, repo_type="model")
        print("uploaded:", repo_path)
    except Exception as exc:
        print("skipped:", local_path, "->", exc)
print("done -- model, config and training log are on the Hub")

## Honest expectations

- 220 minutes on a T4 is a **small token budget** for a 265M-parameter model
  (order 10⁸ tokens). Expect coherent byte-level structure and falling loss,
  not a finished assistant. Continue pre-training by re-running (auto-resume).
- **The mix is a document quota, not a token quota.** 50% PT-BR
  documents does not mean 50% PT-BR bytes (code files are long, captions
  are short); the audit cell prints both shares. Multimodal documents
  also expand into two training rows (caption + media codec).
- **Masking is source-agnostic:** every row trains under the same
  `ignore_index=258` cross-entropy, and a fully masked batch yields a
  zero loss instead of `0/0` NaN. Caption prompts and codec prefixes are
  masked; only the caption/codec targets and EOS are learned.
- COYO images are fetched by URL; dead links are skipped and the mixture
  renormalises. The embedded MSCOCO-5k and LibriSpeech fallbacks keep the
  multimodal fraction alive when COYO fails.
- **Zero-disk streaming is verified, not assumed:** cell 6b prints the
  `IterableDataset` type of every group and asserts ≤ 32 MB disk growth over
  10 streamed batches. If that assert ever fires, the run must not start.
- **The Latent Thought Matrix is a scratchpad, not output.** It never passes
  through the byte head; only the context frontier reads it back (causal).
  Its write gate starts closed (ReZero) and opens during training — cell 8
  prints the activation norm and gates so usage is visible, not assumed.
- The T4 runs **fp16 + GradScaler**, not bf16 — this is resolved and logged at
  runtime, never assumed.
- `[EDIT]` tokens are ordinary vocabulary learned by cross-entropy; only
  `[TOOL_SEARCH]` ids execute anything, inside a resource-limited subprocess.